# Cardiac Patient Monitoring System

## 05 — Feature Engineering and Scikit-learn Pipeline

## Objective

Create meaningful derived features and build a reusable Scikit-learn Pipeline that combines preprocessing and modeling without leaking test information.

## Imports and Load Data

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_PATH = DATA_DIR / "cardio_train.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib

df = pd.read_csv(DATA_DIR / "cardio_clean.csv")


## Feature Engineering

The dataset contains `height` and `weight`, which can be transformed into a BMI-like ratio. We also create a pulse-pressure feature from the two blood-pressure measurements. These are feature transformations for machine-learning analysis, not clinical measures or recommendations.

In [ ]:
def add_features(data):
    data = data.copy()
    data["bmi"] = data["weight"] / ((data["height"] / 100) ** 2)
    data["pulse_pressure"] = data["ap_hi"] - data["ap_lo"]
    return data

df_fe = add_features(df)

print("Original features:", len(df.columns) - 1)
print("Engineered features:", ["bmi", "pulse_pressure"])
display(df_fe[["height", "weight", "ap_hi", "ap_lo", "bmi", "pulse_pressure"]].head())


## Train/Test Split

In [ ]:
X = df_fe.drop(columns=["cardio"])
y = df_fe["cardio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = []

print("Numeric features:", numeric_features)


## Reusable Pipeline

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features)
], remainder="drop")

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
prob = pipeline.predict_proba(X_test)[:, 1]

pipeline_metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred),
    "Recall": recall_score(y_test, pred),
    "F1": f1_score(y_test, pred),
    "ROC-AUC": roc_auc_score(y_test, prob)
}])

display(pipeline_metrics.round(4))


## Save the Pipeline

In [ ]:
pipeline_path = MODEL_DIR / "cardiovascular_logistic_pipeline.joblib"
joblib.dump(pipeline, pipeline_path)
print(f"Pipeline saved to: {pipeline_path}")


## Why a Pipeline?

The Pipeline keeps preprocessing and model training together. This makes the workflow reproducible and helps prevent preprocessing from being fitted separately on the test set.

## Re-run Check

The saved artifact can be loaded later and used on data with the same feature structure.

In [ ]:
loaded_pipeline = joblib.load(pipeline_path)
loaded_pred = loaded_pipeline.predict(X_test)

print("Loaded pipeline predictions match:", np.array_equal(pred, loaded_pred))


## Conclusion

Feature engineering and preprocessing are now encapsulated in a reusable Scikit-learn Pipeline. The next notebook moves from prediction to unsupervised exploration using PCA and clustering.